# Issue [#412](https://github.com/langgraph4j/langgraph4j/issues/412) — interrupt before a long running tool call and resume with its result

[#412]: https://github.com/langgraph4j/langgraph4j/issues/412

The use case: an agent calls a **MCP tool that takes a long time** (minutes). Waiting for it inside the graph
blocks the whole turn, so the application wants to:

1. interrupt the graph **before** the tool runs,
2. execute the tool **outside** the graph,
3. resume the graph **skipping the tool call** and feeding the externally produced result back to the agent.

As suggested by the maintainer in [#412], the ingredients are `approvalOn()` (interrupt before the tool) plus a
custom *approval edge* hook that replaces the built-in APPROVED/REJECTED handling with a `Command` carrying the
external result — the same plumbing the built-in rejection path uses, but with the real tool output.

The notebook uses a **scripted `ChatModel`**, so it runs offline and deterministically. See the last cell for how to
swap in a real model.

In [ ]:
var userHomeDir = System.getProperty("user.home");
var localRespoUrl = "file://" + userHomeDir + "/.m2/repository/";
var langgraph4jVersion = "1.9.5";
var springaiVersion = "2.0.1";

Remove installed package from Jupiter cache

In [ ]:
%%bash
rm -rf \{userHomeDir}/Library/Jupyter/kernels/rapaio-jupyter-kernel/mima_cache/org/bsc/

In [ ]:
%dependency /add-repo local \{localRespoUrl} release|never snapshot|always
%dependency /add org.slf4j:slf4j-jdk14:2.0.9
%dependency /add org.bsc.langgraph4j:langgraph4j-core:\{langgraph4jVersion}
%dependency /add org.bsc.langgraph4j:langgraph4j-spring-ai:\{langgraph4jVersion}
%dependency /add org.bsc.langgraph4j:langgraph4j-springai-agentexecutor:\{langgraph4jVersion}
%dependency /add org.springframework.ai:spring-ai-model:\{springaiVersion}
%dependency /add org.springframework.ai:spring-ai-client-chat:\{springaiVersion}
%dependency /resolve

## Imports

In [ ]:
import java.util.List;
import java.util.Map;
import java.util.concurrent.CompletableFuture;
import java.util.concurrent.atomic.AtomicInteger;
import java.util.concurrent.atomic.AtomicReference;

import org.bsc.langgraph4j.CompileConfig;
import org.bsc.langgraph4j.GraphInput;
import org.bsc.langgraph4j.GraphResult;
import org.bsc.langgraph4j.RunnableConfig;
import org.bsc.langgraph4j.action.Command;
import org.bsc.langgraph4j.action.InterruptionMetadata;
import org.bsc.langgraph4j.agent.AgentEx;
import org.bsc.langgraph4j.checkpoint.MemorySaver;
import org.bsc.langgraph4j.spring.ai.agentexecutor.AgentExecutorEx;
import org.bsc.langgraph4j.state.AgentState;

import org.springframework.ai.chat.messages.AssistantMessage;
import org.springframework.ai.chat.messages.MessageType;
import org.springframework.ai.chat.messages.ToolResponseMessage;
import org.springframework.ai.chat.messages.UserMessage;
import org.springframework.ai.chat.model.ChatModel;
import org.springframework.ai.chat.model.ChatResponse;
import org.springframework.ai.chat.model.Generation;
import org.springframework.ai.chat.prompt.Prompt;
import org.springframework.ai.tool.annotation.Tool;
import org.springframework.ai.tool.annotation.ToolParam;

## The tool and a scripted model

`LongRunningMcpTool` stands in for the remote MCP job: it must **never** be invoked by the graph in this notebook,
which is exactly what the counter proves.

`ScriptedChatModel` replays two canned turns: first it asks for the tool, then — once the conversation carries a
tool response — it answers with it.

In [ ]:
class LongRunningMcpTool {

    final AtomicInteger invocations = new AtomicInteger();

    @Tool(name = "mcp_long_job", description = "A stand-in for a long running MCP tool call")
    public String mcpLongJob(@ToolParam(description = "the query to run") String query) {
        invocations.incrementAndGet();
        return "in-graph execution (must be skipped)";
    }
}

class ScriptedChatModel implements ChatModel {

    @Override
    public ChatResponse call(Prompt prompt) {

        final var hasToolResponse = prompt.getInstructions()
            .stream()
            .anyMatch(message -> MessageType.TOOL == message.getMessageType());

        if (hasToolResponse) {
            return new ChatResponse(List.of(new Generation(
                new AssistantMessage("final answer: the job reported 42 widgets shipped"))));
        }

        final var toolCall = new AssistantMessage.ToolCall("call-1", "function", "mcp_long_job",
                "{\"query\":\"quarterly report\"}");

        return new ChatResponse(List.of(new Generation(
            AssistantMessage.builder().content("").toolCalls(List.of(toolCall)).build())));
    }
}

final var tool = new LongRunningMcpTool();

## Build the agent: interrupt before the tool

`approvalOn("mcp_long_job", ...)` inserts an `approval_mcp_long_job` node in front of the tool node. The first run
stops there and returns `InterruptionMetadata` — the tool has not been called yet.

The `addApprovalActionHook` below runs on **every** resume: when the external result is available it bypasses the
tool node, injects a `ToolResponseMessage` built from that result and routes to the action dispatcher, which is
exactly what the built-in rejection path does for its "denied" message.

In [ ]:
final var externalResult = new AtomicReference<String>();

final var agent = AgentExecutorEx.builder()
    .chatModel(new ScriptedChatModel())
    .toolsFromObject(tool)
    .approvalOn("mcp_long_job", (nodeId, state) -> InterruptionMetadata.builder(nodeId, state)
        .addMetadata("tool_name", "mcp_long_job")
        .addMetadata("job_id", "job-42")
        .build())
    .addApprovalActionHook((sourceId, state, config, action) -> {

        final var result = externalResult.get();
        if (result == null) {
            return action.apply(state, config);
        }

        final var request = state.toolExecutionRequests$getFirst()
            .orElseThrow(() -> new IllegalStateException("no tool execution request found!"));

        final var response = new ToolResponseMessage.ToolResponse(request.id(), request.name(), result);

        return CompletableFuture.completedFuture(new Command(AgentEx.ACTION_DISPATCHER_NODE,
                Map.of(AgentExecutorEx.State.TOOL_EXECUTION_REQUESTS, state.toolExecutionRequests$removeFirst(),
                        AgentExecutorEx.State.TOOL_EXECUTION_RESPONSES, response,
                        AgentEx.APPROVAL_RESULT, AgentState.MARK_FOR_REMOVAL)));
    })
    .build(CompileConfig.builder().checkpointSaver(new MemorySaver()).build());

final var runnableConfig = RunnableConfig.builder().threadId("issue-412").build();

System.out.println(agent.getGraph(org.bsc.langgraph4j.GraphRepresentation.Type.MERMAID, "issue-412", false));

### First run: the graph stops before the tool

In [ ]:
final var interrupted = agent
    .stream(GraphInput.args(Map.of("messages", new UserMessage("run the long MCP job"))), runnableConfig)
    .toCompletableFuture()
    .thenApply(GraphResult::from)
    .join();

System.out.println("interrupted: " + interrupted.isInterruptionMetadata());
System.out.println("job id: " + interrupted.asInterruptionMetadata().metadata("job_id").orElse("<none>"));

// the tool has NOT run
System.out.println("in-graph tool invocations: " + tool.invocations.get());

## Execute the tool outside the graph, then resume

At this point the application would call the real MCP server (another thread, a separate process, an async job — the
transport does not matter) and then resume the graph with the result. Here the "external execution" is just a value.

In [ ]:
// pretend a background worker completed the MCP job
externalResult.set("external MCP job result: 42 widgets shipped");

final var resumed = agent
    .stream(GraphInput.resume(Map.of(AgentEx.APPROVAL_RESULT, AgentEx.ApprovalState.REJECTED)), runnableConfig)
    .toCompletableFuture()
    .thenApply(GraphResult::from)
    .join();

final var state = new AgentExecutorEx.State(resumed.asStateDataOrLastCheckpointStateData());

System.out.println("in-graph tool invocations: " + tool.invocations.get());

System.out.println("tool responses in the conversation:");
state.messages().stream()
    .filter(ToolResponseMessage.class::isInstance)
    .map(ToolResponseMessage.class::cast)
    .flatMap(message -> message.getResponses().stream())
    .forEach(response -> System.out.println("  - " + response.responseData()));

System.out.println("final answer: " + state.lastMessage()
    .filter(AssistantMessage.class::isInstance)
    .map(AssistantMessage.class::cast)
    .map(AssistantMessage::getText)
    .orElse("<none>"));

## Notes

- **Resume value.** The hook intercepts before the built-in edge runs, so the `APPROVAL_RESULT` value itself is not
  meaningful; `REJECTED` is used here to mean "do not run the tool". Without the hook, `APPROVED` would execute the
  tool inside the graph, as `Issue412Test#withoutTheHookApprovedResumeExecutesTheTool` shows.
- **Multiple tool calls.** `state.toolExecutionRequests$removeFirst()` keeps the remaining requests in the state; the
  dispatcher then moves on to the next one. Only the request being resumed is replaced by the external result.
- **Real model.** Replace `ScriptedChatModel` with any Spring AI `ChatModel` (eg. `OpenAiChatModel` or
  `OllamaChatModel`) and the same flow applies — the interruption, the external execution and the resume do not depend
  on the model.